# PC-FNO3D NATIONAL v4 — lecture UNIQUE du test 2022 (Colab, A100)

Règle `configs/national/regle_v4_national.json` (sha256 `ebeb761d3146…`), vérifiée par le carnet. **Uniquement avec l'accord écrit d'Ines.** Vérité d'entraînement v3 (`cache_v3_partie_*`).

La graine est choisie sur le score de validation des trois entraînements ; le test n'est lu que pour elle, une fois.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "c4b49cd135225c47c611eb660dae2467256bc997"
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"], check=True)
os.chdir("/content/PFE")
import hashlib
for f, h in (("regle_v4_national.json", "ebeb761d31462f044eb4acf6b5d5c08b0ec84a7a4d37b6035278f6dc22fe08c3"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

## 2. Cache v3

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 3. Mémoire du GPU

In [ ]:
import torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model

def essai(recalcul: bool) -> float:
    cfg = Config.load("configs/national_1km_colab.yaml")
    cfg.model.recalcul_activations = recalcul
    geom = Geometry(cfg, device="cuda")
    m = build_model(cfg, geom.c_in).cuda().train()
    x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
    del m, x
    torch.cuda.empty_cache()
    return pic

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{total:.0f} Go")
try:
    pic = essai(False)
    RECALCUL = pic > 0.85 * total
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    pic, RECALCUL = float("nan"), True
if RECALCUL:
    pic = essai(True)
print(f"pic {pic:.1f} Go sur {total:.0f} | recalcul des activations : {RECALCUL}")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. Choix de la graine, test, DSO, mesures réelles

In [ ]:
import shlex
from google.colab import runtime
import traceback
regle = json.loads((CODE / "configs" / "national" / "regle_v4_national.json").read_text(encoding="utf-8"))
scores = {}
for g in regle["candidat"]["graines"]:
    h = NAT / "runs" / f"national_1km_v4_C_s{g}" / "history.json"
    if not h.exists():
        raise FileNotFoundError(f"graine {g} pas entrainee ({h})")
    scores[g] = min(float(e["val_score"]) for e in json.loads(h.read_text()) if "val_score" in e)
print("scores de selection (validation) :", scores)
GRAINE = min(scores, key=scores.get)
RUN = NAT / "runs" / f"national_1km_v4_C_s{GRAINE}"
choix = {"regle_sha256": "ebeb761d31462f044eb4acf6b5d5c08b0ec84a7a4d37b6035278f6dc22fe08c3", "scores_validation": scores, "graine_choisie": GRAINE}
(NAT / "runs" / "choix_graine_v4.json").write_text(json.dumps(choix, indent=2))
import yaml
c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))
c["name"] = "national_v4_eval"
c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RUN), f"/content/run_local_v4_s{GRAINE}"
yaml.safe_dump(c, open(CODE / "configs" / "national_v4_eval.yaml", "w"), sort_keys=False)
O = OPTS
try:
    if not (RUN / "eval_pcfno3d" / "best_wcont" / "test_summary.json").exists():   # test lu une fois
        lancer("evaluer_v4", f"python -u -m pcfno3d -c national_v4_eval {O} evaluate")
    CAL = RUN / "fiabilite_pcfno3d" / "best_wcont" / "calibration_validation.json"
    D6 = RUN / "eval_pcfno3d" / "best_wcont" / "dso6_modele_livre.json"
    if not CAL.exists():
        lancer("calibrer_dso6_v4", f"python -u -m pcfno3d -c national_v4_eval {O} calibrate-reliability --out {CAL}")
    if not D6.exists():
        try:
            lancer("dso6_v4", f"python -u -m pcfno3d -c national_v4_eval {O} dso6 --calibration {CAL} --out {D6}")
        except RuntimeError as exc:
            if "(code 3)" not in str(exc):
                raise
            print(exc)
    if not (RUN / "validation_reelle" / "test" / "bilan.json").exists():
        lancer("validation_reelle_v4", f"python -u scripts/valider_grand_tunis_reel.py -c national_v4_eval "
                                       f"--obs {G / 'obs'} --jeu test")
    for cmd, nom in ((f"python -u -m pcfno3d -c national_v4_eval {O} verdict-national", "verdict_national_v4"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v4_eval regle", "dso_sol_regle_v4"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v4_eval mesure", "dso_sol_v4"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v4_eval mesure --keep 0.25", "dso_sol_dso5_v4")):
        try:
            lancer(nom, cmd)
        except RuntimeError as exc:
            if "(code 3)" not in str(exc):
                raise
            print(exc)
    for f in sorted(RUN.rglob("bilan.json")) + sorted(RUN.rglob("verdict_national.json")):
        print(f); print(f.read_text()[:3000])
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()